Bronze layer ingestion for drivers

In [0]:
%python
path = '/Volumes/workspace/streaming_cdc_test/raw_data/drivers'

streamingInputDriversDF = (
    spark.readStream
     .format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("cloudFiles.inferColumnTypes", "true")
     .option("cloudFiles.schemaLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/drivers_schema_bronze')
     .load(path)
)

streamingInputDriversDF.writeStream\
                .option("checkpointLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/bronze_drivers')\
                .outputMode('append')\
                .trigger(availableNow=True)\
                .toTable('workspace.streaming_cdc_test.drivers_bronze')

Bronze layer ingestion for rides

In [0]:
%python
path = '/Volumes/workspace/streaming_cdc_test/raw_data/rides'

streamingInputRidesDF = (
    spark.readStream
     .format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("cloudFiles.inferColumnTypes", "true")
     .option("cloudFiles.schemaLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/rides_schema_bronze')
     .load(path)
)

streamingInputRidesDF.writeStream\
                .option("checkpointLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/bronze_rides')\
                .outputMode('append')\
                .trigger(availableNow=True)\
                .toTable('workspace.streaming_cdc_test.rides_bronze')


In [0]:
%python
#I needed it to clean the wrong schema
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/bronze', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/schema_bronze', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/drivers', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/rides', True)

In [0]:
--I needed it to clean the wrong schema
DROP TABLE IF EXISTS workspace.streaming_cdc_test.drivers_bronze

Silver layer transformations for drivers

In [0]:
%python
import pyspark.sql.functions as F

def merge_drivers(df, batch_id):
    df_dedup = df.dropDuplicates(['id'])
    df_dedup.createOrReplaceTempView('drivers_micro_df')
    sql_query = """
        MERGE INTO workspace.streaming_cdc_test.drivers_silver t
        USING drivers_micro_df s
        ON t.id = s.id
        WHEN MATCHED THEN
        UPDATE SET *
        WHEN NOT MATCHED
        THEN INSERT *
    """
    df_dedup.sparkSession.sql(sql_query)

spark.sql("""
          CREATE TABLE IF NOT EXISTS workspace.streaming_cdc_test.drivers_silver(
              id STRING,
              first_name STRING,
              last_name STRING,
              car_number STRING,
              experience INTEGER,
              rating INTEGER
          )
          TBLPROPERTIES (delta.enableChangeDataFeed = true)""")

streamingInputDriversDF = (spark.readStream
                        .table('workspace.streaming_cdc_test.drivers_bronze')
                        .withColumn('first_name', F.split(F.col('name'), ' ')[0])
                        .withColumn('last_name',  F.split(F.col('name'), ' ')[1])
                        .select('id', 'first_name', 'last_name', 'car_number', 'experience', 'rating')
)

(
    streamingInputDriversDF.writeStream
                    .format('delta')
                    .option('checkpointLocation', '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/silver_drivers')
                    .outputMode("append")
                    .foreachBatch(merge_drivers)
                    .trigger(availableNow=True)
                    .start()
)
     

In [0]:
%python
def merge_riders(df, batch_id):
    df_dedup = df.dropDuplicates(['driver_id'])
    df_drivers = df.sparkSession.read.table('workspace.streaming_cdc_test.drivers_silver')
    df_dedup = df_dedup.join(df_drivers, df_drivers.id == df_dedup.driver_id, "semi")
    df_dedup.createOrReplaceTempView('riders_micro_df')

    sql_query = """
        MERGE INTO workspace.streaming_cdc_test.rides_silver t
        USING riders_micro_df s
        ON t.ride_id = s.ride_id
        WHEN MATCHED THEN
        UPDATE SET *
        WHEN NOT MATCHED
        THEN INSERT *
    """
    df_dedup.sparkSession.sql(sql_query)

spark.sql("""
          CREATE TABLE IF NOT EXISTS workspace.streaming_cdc_test.rides_silver(
              ride_id STRING,
              driver_id STRING,
              distance INTEGER,
              cost INTEGER
          )
          TBLPROPERTIES (delta.enableChangeDataFeed = true)""")

streamingInputRidesDF = (
    spark.readStream
         .table('workspace.streaming_cdc_test.rides_bronze')
         .select('ride_id', 'driver_id', 'distance', 'cost')
)

(
    streamingInputRidesDF.writeStream
                    .format('delta')
                    .option('checkpointLocation', '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/silver_rides')
                    .outputMode("append")
                    .foreachBatch(merge_riders)
                    .trigger(availableNow=True)
                    .start()
)